In [2]:
cd "C:\Users\demen\OneDrive\ERCS_SD_Bot"

C:\Users\demen\OneDrive\ERCS_SD_Bot


C:\Users\demen\AppData\Roaming\Python\Python312\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
pip install --upgrade python-telegram-bot

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Step 1: Install required packages (run this only once)
!pip install python-telegram-bot --quiet
!pip install nest_asyncio --quiet

In [5]:
# Step 2: Run this to set up the bot

import nest_asyncio
import asyncio
from telegram import Update
from telegram.ext import (
    Application,
    CommandHandler,
    ContextTypes,
)
from telegram.request import HTTPXRequest

# This allows async code to run in Jupyter
nest_asyncio.apply()

# Define the command handler
async def contacts_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    contact_info = (
        "📞 *Contacts:*\n"
        "Email: demmenew1@gmail,com\n"
        "Phone: +251-916-866654\n"
    await update.message.reply_markdown(contact_info)

# Replace with your real, secure token from BotFather
BOT_TOKEN = "YOUR_NEW_BOT_TOKEN_HERE"  # ← 🔐 Insert your token here

# Build the application
request = HTTPXRequest(connect_timeout=10.0)
application = Application.builder().token(BOT_TOKEN).request(request).build()

# Add the /contacts command handler
application.add_handler(CommandHandler("contacts", contacts_command))

# Run the bot in Jupyter
async def run_bot():
    print("✅ ERCS Bot is running. Try sending /contacts to @ERCS_SD_bot on Telegram.")
    await application.initialize()
    await application.start()
    await application.updater.start_polling()

# Start the bot
await run_bot()

SyntaxError: '(' was never closed (3281470683.py, line 18)

In [ ]:
# ========== STEP 1: SETUP AND IMPORTS ==========
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, ContextTypes, CommandHandler,
    MessageHandler, filters
)
import nest_asyncio
nest_asyncio.apply()  # Required for Jupyter Notebooks

# ========== STEP 2: BOT CONFIGURATION ==========
TOKEN = "7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs"  # Replace with your actual token
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID

# Save location for reports (OneDrive-synced folder)
REPORT_FOLDER = r"C:\Users\demen\OneDrive\ERCS_SD_Bot"
os.makedirs(REPORT_FOLDER, exist_ok=True)

# ========== STEP 3: LANGUAGE & MENU ==========
LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}  # User session storage
all_reports = []  # List to store reports

DISASTER_TYPES = [
    ("Conflict", "💥"),
    ("Flood", "🌊"),
    ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"),
    ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"),
    ("Disease", "💉"),
    ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"),
    ("Cholera", "💩"),
    ("Measles", "🤒"),
    ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Wonosho", "Yirgalem town"
]

# ========== STEP 4: REPORT GENERATION ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== STEP 5: START & MENU ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True)
    )

# ========== STEP 6: MESSAGE HANDLER ==========
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    # Language switch
    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await show_main_menu(update, user_id)
        return

    # Main menu options
    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    # Step-by-step handling
    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)

# ========== STEP 7: REPORT FLOW ==========
async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], resize_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    if user_data[user_id]['form']['Disaster Type'] == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], resize_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], resize_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    user_data[user_id]['form']['Woreda'] = msg
    user_data[user_id]['step'] = 'kebele'
    await update.message.reply_text(
        "Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:"
    )

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text(
        "Enter number of affected people:" if lang == 'en' else "ተጎዳኙትን ብዛት ያስገቡ:"
    )

async def request_deaths(update, user_id, lang, msg):
    user_data[user_id]['form']['Affected'] = msg
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text(
        "Enter number of deaths:" if lang == 'en' else "የሞቱትን ብዛት ያስገቡ:"
    )

async def request_suggestions(update, user_id, lang, msg):
    user_data[user_id]['form']['Deaths'] = msg
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text(
        "Any suggestions for response?" if lang == 'en' else "ምንም ዓይነት የመልስ አማራጭ አለ?"
    )

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    generate_weekly_report()

    await update.message.reply_text(
        "✅ Report submitted successfully!" if lang == 'en' else "✅ ሪፖርቱ በትክክል ተሰጥቷል!"
    )
    await show_main_menu(update, user_id)

# ========== STEP 8: FEEDBACK ==========
async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text(
        "Please enter your comment/suggestion:" if lang == 'en'
        else "እባክዎን አስተያየትዎን ያስገቡ።"
    )

# ========== STEP 9: LAUNCH BOT ==========
def main():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("🤖 Bot is running. Press Ctrl+C to stop.")
    app.run_polling()

if __name__ == '__main__':
    main()

🤖 Bot is running. Press Ctrl+C to stop.


C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\selectors.py:298: RuntimeWarning: coroutine 'Application.shutdown' was never awaited
  def register(self, fileobj, events, data=None):
C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\selectors.py:298: RuntimeWarning: coroutine 'Application._bootstrap_initialize' was never awaited
  def register(self, fileobj, events, data=None):


In [ ]:
from telegram.request import HTTPXRequest

request = HTTPXRequest(connect_timeout=10.0)  # default is 5 seconds
application = Application.builder().token(7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs).request(request).build()

In [9]:
# pip install --upgrade python-telegram-bot

# ========== STEP 1: SETUP AND IMPORTS ==========
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, ContextTypes, CommandHandler,
    MessageHandler, filters
)
import nest_asyncio
nest_asyncio.apply()

# ========== STEP 2: BOT CONFIGURATION ==========
TOKEN = "7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs"  # Replace with your actual token
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID

REPORT_FOLDER = r"C:\Users\demen\OneDrive\ERCS_SD_Bot"
os.makedirs(REPORT_FOLDER, exist_ok=True)

# ========== STEP 3: LANGUAGE & MENU ==========
LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}
all_reports = []

DISASTER_TYPES = [
    ("Conflict", "💥"), ("Flood", "🌊"), ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"), ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"), ("Disease", "💉"), ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"), ("Cholera", "💩"), ("Measles", "🤒"), ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Bura", "Bursa", "Chabe Gambeltu", "Chire", "Chirone", "Chuko town",
    "Daella", "Dale", "Dara", "Dara Otilicho", "Darara", "Daye town",
    "Gorche", "Hawasa town", "Hawassa Zuria", "Hawela", "Hokko", "Hulla",
    "Leku town", "Loka Abaya", "Malga", "Shafamo", "Shebe Dino", "Teticha",
    "Wondo-Genet", "Wondo-Genet town", "Wonosho", "Yirgalem town"
]

# ========== STEP 4: REPORT GENERATION ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== STEP 5: START & MENU ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True)
    )

# ========== STEP 6: MESSAGE HANDLER ==========
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await show_main_menu(update, user_id)
        return

    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)

# ========== STEP 7: REPORT FLOW ==========
async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], resize_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    if user_data[user_id]['form']['Disaster Type'] == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], resize_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], resize_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    user_data[user_id]['form']['Woreda'] = msg
    user_data[user_id]['step'] = 'kebele'
    await update.message.reply_text("Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:")

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text("Enter number of affected people:" if lang == 'en' else "የተጎጅ ብዛት ያስገቡ:")

async def request_deaths(update, user_id, lang, msg):
    user_data[user_id]['form']['Affected'] = msg
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text("Enter number of deaths:" if lang == 'en' else "የሞቱትን ብዛት ያስገቡ:")

async def request_suggestions(update, user_id, lang, msg):
    user_data[user_id]['form']['Deaths'] = msg
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Any suggestions for response?" if lang == 'en' else "ምን ዓይነት ድጋፍ ያስፈልጋል?")

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    generate_weekly_report()
    await update.message.reply_text("✅ Report submitted successfully!" if lang == 'en' else "✅ ሪፖርቱ በትክክል ተሰጥቷል!")
    await show_main_menu(update, user_id)

# ========== STEP 8: FEEDBACK ==========
async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Please enter your comment/suggestion:" if lang == 'en' else "እባክዎን አስተያየትዎን ያስገቡ።")

# ========== STEP 9: LAUNCH BOT ==========
def main():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("🤖 Bot is running. Press Ctrl+C to stop.")
    app.run_polling()

if __name__ == '__main__':
    main()


🤖 Bot is running. Press Ctrl+C to stop.


RuntimeError: Cannot close a running event loop

In [1]:
# ========== SETUP ==========
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, CommandHandler, MessageHandler,
    ContextTypes, filters
)

# ========== CONFIG ==========
TOKEN = "7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs"  # Replace with your actual secure bot token
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID

REPORT_FOLDER = "/home/YOUR_USERNAME/ercs_reports"  # ← Make sure this folder exists
os.makedirs(REPORT_FOLDER, exist_ok=True)

LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}
all_reports = []

DISASTER_TYPES = [
    ("Conflict", "💥"),
    ("Flood", "🌊"),
    ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"),
    ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"),
    ("Disease", "💉"),
    ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"),
    ("Cholera", "💩"),
    ("Measles", "🤒"),
    ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Wonosho", "Yirgalem town"
]

# ========== REPORT ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== BOT LOGIC ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True)
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await show_main_menu(update, user_id)
        return

    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)

async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], resize_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    if user_data[user_id]['form']['Disaster Type'] == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], resize_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], resize_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    user_data[user_id]['form']['Woreda'] = msg
    user_data[user_id]['step'] = 'kebele'
    await update.message.reply_text("Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:")

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text("Number affected:" if lang == 'en' else "የተጎጅ ብዛት:")

async def request_deaths(update, user_id, lang, msg):
    user_data[user_id]['form']['Affected'] = msg
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text("Number of deaths:" if lang == 'en' else "የሞቱት ብዛት:")

async def request_suggestions(update, user_id, lang, msg):
    user_data[user_id]['form']['Deaths'] = msg
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Any response suggestion?" if lang == 'en' else "የሚፈለገው ድጋፍ?")

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    generate_weekly_report()
    await update.message.reply_text("✅ Report submitted!" if lang == 'en' else "✅ ሪፖርት ተሰጥቷል!")
    await show_main_menu(update, user_id)

async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Your comment?" if lang == 'en' else "አስተያየትዎን ያስገቡ።")

# ========== MAIN ==========
def main():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("🤖 Bot is running...")
    app.run_polling()

if __name__ == '__main__':
    main()

🤖 Bot is running...


RuntimeError: Cannot close a running event loop

In [ ]:
pip install pandas

In [ ]:
pip3 install pandas

In [ ]:
!pip install pandas

In [ ]:
# On Windows
.\venv\Scripts\activate

# On Mac/Linux
source venv/bin/activate

pip install pandas

In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
pip install --upgrade python-telegram-bot

In [ ]:
    pip install --upgrade python-telegram-bot

In [ ]:
# ========== SETUP ==========
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, CommandHandler, MessageHandler,
    ContextTypes, filters
)

# ========== CONFIG ==========
TOKEN = "7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs"  # Replace with your actual secure bot token
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID

REPORT_FOLDER = "/home/YOUR_USERNAME/ercs_reports"  # ← Make sure this folder exists
os.makedirs(REPORT_FOLDER, exist_ok=True)

LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}
all_reports = []

DISASTER_TYPES = [
    ("Conflict", "💥"),
    ("Flood", "🌊"),
    ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"),
    ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"),
    ("Disease", "💉"),
    ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"),
    ("Cholera", "💩"),
    ("Measles", "🤒"),
    ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Bura", "Bursa", "Chabe Gambeltu", "Chire", "Chirone", "Chuko town",
    "Daella", "Dale", "Dara", "Dara Otilicho", "Darara", "Daye town",
    "Gorche", "Hawasa town", "Hawassa Zuria", "Hawela", "Hokko", "Hulla",
    "Leku town", "Loka Abaya", "Malga", "Shafamo", "Shebe Dino", "Teticha",
    "Wondo-Genet", "Wondo-Genet town", "Wonosho", "Yirgalem town"
]

# ========== REPORT ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== BOT LOGIC ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True)
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await show_main_menu(update, user_id)
        return

    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)

async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], resize_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    if user_data[user_id]['form']['Disaster Type'] == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], resize_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], resize_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    user_data[user_id]['form']['Woreda'] = msg
    user_data[user_id]['step'] = 'kebele'
    await update.message.reply_text("Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:")

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text("Number affected:" if lang == 'en' else "የተጎጅ ብዛት:")

async def request_deaths(update, user_id, lang, msg):
    user_data[user_id]['form']['Affected'] = msg
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text("Number of deaths:" if lang == 'en' else "የሞቱት ብዛት:")

async def request_suggestions(update, user_id, lang, msg):
    user_data[user_id]['form']['Deaths'] = msg
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Any response suggestion?" if lang == 'en' else "የሚፈለገው ድጋፍ?")

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    generate_weekly_report()
    await update.message.reply_text("✅ Report submitted!" if lang == 'en' else "✅ ሪፖርት ተሰጥቷል!")
    await show_main_menu(update, user_id)

async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Your comment?" if lang == 'en' else "አስተያየትዎን ያስገቡ።")

# ========== MAIN ==========
def main():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("🤖 Bot is running...")
    app.run_polling()

if __name__ == '__main__':
    main()

In [ ]:
import requests
PA_USERNAME = 'demelash'
PA_TOKEN = 'fa7a65ed8145a9999fb84765b22bc99c4f71fcea'

# ========== SETUP ==========
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, CommandHandler, MessageHandler,
    ContextTypes, filters
)

# ========== CONFIG ==========
TOKEN = "7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs"  # Replace with your actual secure bot token
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID

REPORT_FOLDER = "/home/YOUR_USERNAME/ercs_reports"  # ← Make sure this folder exists
os.makedirs(REPORT_FOLDER, exist_ok=True)

LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}
all_reports = []

DISASTER_TYPES = [
    ("Conflict", "💥"),
    ("Flood", "🌊"),
    ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"),
    ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"),
    ("Disease", "💉"),
    ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"),
    ("Cholera", "💩"),
    ("Measles", "🤒"),
    ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Bura", "Bursa", "Chabe Gambeltu", "Chire", "Chirone", "Chuko town",
    "Daella", "Dale", "Dara", "Dara Otilicho", "Darara", "Daye town",
    "Gorche", "Hawasa town", "Hawassa Zuria", "Hawela", "Hokko", "Hulla",
    "Leku town", "Loka Abaya", "Malga", "Shafamo", "Shebe Dino", "Teticha",
    "Wondo-Genet", "Wondo-Genet town", "Wonosho", "Yirgalem town"
]

# ========== REPORT ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== BOT LOGIC ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True)
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await show_main_menu(update, user_id)
        return

    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)

async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], resize_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    if user_data[user_id]['form']['Disaster Type'] == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], resize_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], resize_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    user_data[user_id]['form']['Woreda'] = msg
    user_data[user_id]['step'] = 'kebele'
    await update.message.reply_text("Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:")

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text("Number affected:" if lang == 'en' else "የተጎጅ ብዛት:")

async def request_deaths(update, user_id, lang, msg):
    user_data[user_id]['form']['Affected'] = msg
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text("Number of deaths:" if lang == 'en' else "የሞቱት ብዛት:")

async def request_suggestions(update, user_id, lang, msg):
    user_data[user_id]['form']['Deaths'] = msg
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Any response suggestion?" if lang == 'en' else "የሚፈለገው ድጋፍ?")

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    generate_weekly_report()
    await update.message.reply_text("✅ Report submitted!" if lang == 'en' else "✅ ሪፖርት ተሰጥቷል!")
    await show_main_menu(update, user_id)

async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text("Your comment?" if lang == 'en' else "አስተያየትዎን ያስገቡ።")

# ========== MAIN ==========
def main():
    app = ApplicationBuilder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    print("🤖 Bot is running...")
    app.run_polling()

if __name__ == '__main__':
    main()

In [4]:
import os
import pandas as pd
from datetime import datetime
from telegram import Update, ReplyKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, ContextTypes, CommandHandler,
    MessageHandler, filters
)
import nest_asyncio
nest_asyncio.apply()  # Required for Jupyter Notebooks

# ========== BOT CONFIGURATION ==========
TOKEN = os.getenv("7888149981:AAFrdXdYMPLYjz-F-jYQNOwSFWb4-PdfAHs", "YOUR_TOKEN_HERE")  # Use environment variable for security
ADMIN_IDS = [123456789]  # Replace with your Telegram user ID
REPORT_FOLDER = "reports"  # Simplified folder path
os.makedirs(REPORT_FOLDER, exist_ok=True)

# ========== LANGUAGE & MENU ==========
LANGUAGES = {'en': 'English', 'am': 'Amharic'}
user_data = {}  # User session storage
all_reports = []  # List to store reports

DISASTER_TYPES = [
    ("Conflict", "💥"),
    ("Flood", "🌊"),
    ("Fire", "🔥"),
    ("Road Traffic Accident", "🚗💥"),
    ("Landslide", "🌍🌿"),
    ("Earthquake", "🌍⚡"),
    ("Disease", "💉"),
    ("Others", "❓")
]

DISEASE_TYPES = [
    ("Malaria", "🦟"),
    ("Cholera", "💩"),
    ("Measles", "🤒"),
    ("Others", "❓")
]

WOREDA_OPTIONS = [
    "Aleta Chuko", "Aleta Wendo", "Aleta Wondo town", "Arbegona",
    "Aroresa", "Bensa", "Bilate Zuria", "Bona Zuria", "Boricha",
    "Bura", "Bursa", "Chabe Gambeltu", "Chire", "Chirone", "Chuko town",
    "Daella", "Dale", "Dara", "Dara Otilicho", "Darara", "Daye town",
    "Gorche", "Hawasa town", "Hawassa Zuria", "Hawela", "Hokko", "Hulla",
    "Leku town", "Loka Abaya", "Malga", "Shafamo", "Shebe Dino", "Teticha",
    "Wondo-Genet", "Wondo-Genet town", "Wonosho", "Yirgalem town"
]

# ========== REPORT GENERATION ==========
def generate_weekly_report():
    if not all_reports:
        return None
    df = pd.DataFrame(all_reports)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Week'] = df['Timestamp'].dt.isocalendar().week
    filename = os.path.join(
        REPORT_FOLDER, f"Emergency_Report_Week_{datetime.now().strftime('%W')}.xlsx"
    )
    df.to_excel(filename, index=False)
    return filename

# ========== BOT HANDLERS ==========
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id not in user_data:
        user_data[user_id] = {'lang': 'en'}
    await show_main_menu(update, user_id)

async def show_main_menu(update, user_id):
    lang = user_data[user_id].get('lang', 'en')
    menu_items = [
        ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"],
        ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"],
        ["🌐 Language", "🌐 ቋንቋ"]
    ]
    menu = [[item[0] if lang == 'en' else item[1]] for item in menu_items]
    await update.message.reply_text(
        "Please select an option:" if lang == 'en' else "እባክዎን አንዱን ይምረጡ።",
        reply_markup=ReplyKeyboardMarkup(menu, resize_keyboard=True, one_time_keyboard=True)
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    msg = update.message.text

    if user_id not in user_data:
        await start(update, context)
        return

    lang = user_data[user_id].get('lang', 'en')

    # Language switch
    if msg in ["🌐 Language", "🌐 ቋንቋ"]:
        user_data[user_id]['lang'] = 'am' if lang == 'en' else 'en'
        await update.message.reply_text(
            "Language switched to English" if user_data[user_id]['lang'] == 'en' else "ቋንቋ ወደ አማርኛ ተቀይሯል"
        )
        await show_main_menu(update, user_id)
        return

    # Main menu options
    if msg in ["🆘 Emergency Report", "🆘 ድንገተኛ ክስተት ሪፖርት"]:
        await start_emergency_report(update, user_id, lang)
        return
    if msg in ["💬 Suggestions/Feedback", "💬 አስተያየቶች/ግብረመልስ"]:
        await start_feedback(update, user_id, lang)
        return

    # Step-by-step handling
    step = user_data[user_id].get('step')
    if step == 'disaster_type':
        await process_disaster_type(update, user_id, lang, msg)
    elif step == 'disease_type':
        await process_disease_type(update, user_id, lang, msg)
    elif step == 'woreda':
        await request_kebele(update, user_id, lang, msg)
    elif step == 'kebele':
        await request_affected(update, user_id, lang, msg)
    elif step == 'affected':
        await request_deaths(update, user_id, lang, msg)
    elif step == 'dead':
        await request_suggestions(update, user_id, lang, msg)
    elif step == 'suggestions':
        await complete_report(update, user_id, lang, msg)
    elif step == 'feedback':
        await save_feedback(update, user_id, lang, msg)

# ========== REPORT FLOW ==========
async def start_emergency_report(update, user_id, lang):
    user_data[user_id]['step'] = 'disaster_type'
    user_data[user_id]['form'] = {'User ID': user_id}
    keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISASTER_TYPES], 
                                  resize_keyboard=True, 
                                  one_time_keyboard=True)
    await update.message.reply_text(
        "Select Disaster Type:" if lang == 'en' else "የአደጋውን አይነት ይምረጡ:",
        reply_markup=keyboard
    )

async def process_disaster_type(update, user_id, lang, msg):
    for disaster, symbol in DISASTER_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disaster Type'] = disaster
            break
    
    if user_data[user_id]['form'].get('Disaster Type') == "Disease":
        user_data[user_id]['step'] = 'disease_type'
        keyboard = ReplyKeyboardMarkup([[f"{d[1]} {d[0]}"] for d in DISEASE_TYPES], 
                                      resize_keyboard=True,
                                      one_time_keyboard=True)
        await update.message.reply_text(
            "Select Disease Type:" if lang == 'en' else "የበሽታውን አይነት ይምረጡ:",
            reply_markup=keyboard
        )
    else:
        user_data[user_id]['form']['Disease'] = 'N/A'
        await request_woreda(update, user_id, lang)

async def process_disease_type(update, user_id, lang, msg):
    for disease, symbol in DISEASE_TYPES:
        if msg.startswith(symbol):
            user_data[user_id]['form']['Disease'] = disease
            break
    await request_woreda(update, user_id, lang)

async def request_woreda(update, user_id, lang):
    user_data[user_id]['step'] = 'woreda'
    keyboard = ReplyKeyboardMarkup([[w] for w in WOREDA_OPTIONS], 
                                  resize_keyboard=True,
                                  one_time_keyboard=True)
    await update.message.reply_text(
        "Select Woreda:" if lang == 'en' else "ወረዳ ይምረጡ:",
        reply_markup=keyboard
    )

async def request_kebele(update, user_id, lang, msg):
    if msg in WOREDA_OPTIONS:
        user_data[user_id]['form']['Woreda'] = msg
        user_data[user_id]['step'] = 'kebele'
        await update.message.reply_text(
            "Enter Kebele:" if lang == 'en' else "ቀበሌ ያስገቡ:",
            reply_markup=ReplyKeyboardMarkup([["Skip"]], resize_keyboard=True)
        )
    else:
        await update.message.reply_text("Invalid selection. Please choose from the options.")

async def request_affected(update, user_id, lang, msg):
    user_data[user_id]['form']['Kebele'] = msg if msg != "Skip" else "N/A"
    user_data[user_id]['step'] = 'affected'
    await update.message.reply_text(
        "Enter number of affected people:" if lang == 'en' else "የተጎጅ ብዛት ያስገቡ:",
        reply_markup=ReplyKeyboardMarkup([["Unknown"]], resize_keyboard=True)
    )

async def request_deaths(update, user_id, lang, msg):
    try:
        int(msg)  # Validate number
        user_data[user_id]['form']['Affected'] = msg
    except:
        user_data[user_id]['form']['Affected'] = msg  # Store "Unknown"
        
    user_data[user_id]['step'] = 'dead'
    await update.message.reply_text(
        "Enter number of deaths:" if lang == 'en' else "የሞቱት ብዛት ያስገቡ:",
        reply_markup=ReplyKeyboardMarkup([["Unknown"]], resize_keyboard=True)
    )

async def request_suggestions(update, user_id, lang, msg):
    try:
        int(msg)  # Validate number
        user_data[user_id]['form']['Deaths'] = msg
    except:
        user_data[user_id]['form']['Deaths'] = msg  # Store "Unknown"
        
    user_data[user_id]['step'] = 'suggestions'
    await update.message.reply_text(
        "Any suggestions for response?" if lang == 'en' else "ምን ዓይነት ድጋፍ ይደረግ?",
        reply_markup=ReplyKeyboardMarkup([["None"]], resize_keyboard=True)
    )

async def complete_report(update, user_id, lang, msg):
    user_data[user_id]['form']['Suggestions'] = msg
    user_data[user_id]['form']['Timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_reports.append(user_data[user_id]['form'])
    
    # Save individual report
    df = pd.DataFrame([user_data[user_id]['form']])
    individual_file = os.path.join(REPORT_FOLDER, f"report_{user_id}_{datetime.now().strftime('%Y%m%d%H%M%S')}.csv")
    df.to_csv(individual_file, index=False)
    
    # Generate weekly report
    generate_weekly_report()

    await update.message.reply_text(
        "✅ Report submitted successfully!" if lang == 'en' else "✅ ሪፖርቱ በትክክል ተሰጥቷል!"
    )
    await show_main_menu(update, user_id)

# ========== FEEDBACK HANDLER ==========
async def start_feedback(update, user_id, lang):
    user_data[user_id]['step'] = 'feedback'
    await update.message.reply_text(
        "Please enter your comment/suggestion:" if lang == 'en'
        else "እባክዎን አስተያየትዎን ያስገቡ።",
        reply_markup=ReplyKeyboardMarkup([["Cancel"]], resize_keyboard=True)
    )

async def save_feedback(update, user_id, lang, msg):
    if msg != "Cancel":
        feedback_file = os.path.join(REPORT_FOLDER, "feedback.txt")
        with open(feedback_file, "a", encoding="utf-8") as f:
            f.write(f"{datetime.now()} - User {user_id}: {msg}\n")
            
        await update.message.reply_text(
            "📝 Feedback received. Thank you!" if lang == 'en'
            else "📝 አስተያየትዎ ተቀብሏል። አመሰግናለሁ!"
        )
    await show_main_menu(update, user_id)

# ========== ADMIN COMMANDS ==========
async def get_report(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    if user_id in ADMIN_IDS:
        filename = generate_weekly_report()
        if filename:
            await update.message.reply_document(document=open(filename, 'rb'))
        else:
            await update.message.reply_text("No reports available yet.")
    else:
        await update.message.reply_text("⛔ Admin access required")

# ========== ERROR HANDLER ==========
async def error_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f"Update {update} caused error: {context.error}")
    await update.message.reply_text("⚠️ An error occurred. Please try again or use /start")

# ========== LAUNCH BOT ==========
def main():
    # Create application
    app = ApplicationBuilder().token(TOKEN).build()
    
    # Add handlers
    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("report", get_report))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    
    # Error handler
    app.add_error_handler(error_handler)
    
    # Start bot
    print("🤖 Emergency Reporting Bot is running...")
    print(f"📁 Reports saving to: {os.path.abspath(REPORT_FOLDER)}")
    app.run_polling()

if __name__ == '__main__':
    main()

🤖 Emergency Reporting Bot is running...
📁 Reports saving to: C:\Users\demen\OneDrive\ERCS_SD_Bot\reports


Network Retry Loop (Bootstrap Initialize Application): Invalid token. Aborting retry loop.
Traceback (most recent call last):
  File "C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\site-packages\telegram\_bot.py", line 844, in initialize
    await self.get_me()
  File "C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\site-packages\telegram\ext\_extbot.py", line 1980, in get_me
    return await super().get_me(
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\site-packages\telegram\_bot.py", line 975, in get_me
    result = await self._post(
             ^^^^^^^^^^^^^^^^^
  File "C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\site-packages\telegram\_bot.py", line 697, in _post
    return await self._do_post(
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\demen\anaconda3\envs\ERCS_SD_Bot\Lib\site-packages\telegram\ext\_extbot.py", line 369, in _do_post
    return await super()._do_post(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\demen\anaconda3\

RuntimeError: Cannot close a running event loop

In [2]:
pip install python-telegram-bot pandas nest-asyncio

Note: you may need to restart the kernel to use updated packages.


In [3]:
python emergency_bot.py

SyntaxError: invalid syntax (907945131.py, line 1)